# Official XFeat — cosine + MNN

Standalone notebook: Run all in a **fresh CPU Colab runtime**. No other notebook is required.

- Fixed conditions preserved: seed 7, up to 800 KP/image, 64 dimensions, 2 CPU threads, warmup 3, repeats 12.
- Same COCO split/synthetic warps and 24 HPatches pairs as the source experiment.
- Accuracy: pooled precision at forward homography error <3 px in resized coordinates, match counts and median error.
- Speed: cached matching, encoding-inclusive matching, and extraction-inclusive full pipeline, measured separately.
- CSV/JSON automatically saved under `Colab Notebooks/success-direct-thermometer-AVX2-SIMD4/results/official_XFeat/<run timestamp>/`. Raw timing samples, CPU and XFeat source hashes are recorded.
- Independent Colab runs may use different CPUs or runtime loads. Check recorded hardware, source hashes and descriptor fingerprints before comparing.

Direct calls to official `xfeat.match(..., min_cossim=-1)` and `xfeat.match_xfeat(...)`. No custom float matcher or thermometer method is executed.

## Previously observed results (reference only)

Recorded from the previously shared combined-run result; **these are not a new execution of this standalone notebook**.

| Dataset | Precision | Encoding-inclusive matching time |
|---|---:|---:|
| Synthetic | 68.43% | 5.035 ms |
| HPatches | 67.30% | 11.794 ms |

The earlier timing run showed substantial variability (including cached/total inconsistencies). Use the new CSV/JSON results to reassess speed; these reference times do not establish a stable ranking.


In [1]:
BINS = None
EXPERIMENT = 'official_XFeat'


In [2]:
!pip -q install opencv-python-headless numba pandas

import os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
import time
import io
import zipfile
import subprocess
import shutil
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from numba import njit, prange, set_num_threads, get_num_threads
from google.colab import files, drive

SEED = 7
TOP_K = 800
LEVELS = BINS or 8
CALIBRATION_WARPS_PER_IMAGE = 6
VALIDATION_WARPS_PER_IMAGE = 3
MAX_POSITIVES_PER_PAIR = 500
GEOMETRIC_THRESHOLD = 3.0
NUMBA_THREADS = 2
RUN_HPatches = True
TIMING_PAIRS_PER_DATASET = 5  # evenly spaced pairs; compare every method on each pair
WARMUP = 3
REPEATS = 12

set_num_threads(NUMBA_THREADS)
torch.set_num_threads(NUMBA_THREADS)
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = 'cpu'  # identical CPU execution for official and proposed pipelines

torch.set_num_interop_threads(1)  # fresh runtime required
xfeat = torch.hub.load('verlab/accelerated_features', 'XFeat', pretrained=True, top_k=TOP_K).to(device).eval()
xfeat.dev = torch.device(device)  # official preprocessing follows this device
for parameter in xfeat.parameters():
    parameter.requires_grad_(False)
print('Device:', device)
# COCO is used for calibration and an internal synthetic-warp check.
# No projection or threshold learning. HPatches is reserved for the external real-photo evaluation at the end.
DATA_SOURCE = 'coco'  # change to 'upload' to use your own JPG/PNG files
COCO_SOURCE_COUNT = 24
COCO_URL = 'https://images.cocodataset.org/zips/val2017.zip'
drive.mount('/content/drive')
data_dir = Path('/content/drive/MyDrive/Colab Notebooks/CSE527_datasets')
data_dir.mkdir(parents=True, exist_ok=True)
RESULT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/success-direct-thermometer-AVX2-SIMD4/results')
print('Persistent dataset cache:', data_dir)

def download_archive(url, destination):
    if destination.exists() and zipfile.is_zipfile(destination):
        print('Using persistent Drive cache:', destination)
        return
    # Reuse a download from an earlier run in this same Colab session, if present.
    legacy = Path('/content/xfeat_data') / destination.name
    if legacy.exists() and zipfile.is_zipfile(legacy):
        shutil.copy2(legacy, destination)
        print('Copied previous runtime download into Drive:', destination)
        return
    print('Downloading once into Drive:', url)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', str(destination), url], check=True)
    if not zipfile.is_zipfile(destination):
        raise RuntimeError(f'Incomplete or invalid ZIP: {destination}. Delete it and rerun this cell.')

if DATA_SOURCE == 'coco':
    coco_zip = data_dir / 'val2017.zip'
    download_archive(COCO_URL, coco_zip)
    with zipfile.ZipFile(coco_zip) as archive:
        names = sorted(n for n in archive.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
        if len(names) < COCO_SOURCE_COUNT:
            raise RuntimeError('COCO archive has fewer images than requested.')
        selected = np.random.default_rng(SEED).choice(len(names), COCO_SOURCE_COUNT, replace=False)
        image_items = [(Path(names[i]).name, archive.read(names[i])) for i in selected]
    print(f'Selected {len(image_items)} COCO images using seed {SEED}; no COCO annotations needed.')
elif DATA_SOURCE == 'upload':
    print('Upload JPG/PNG images from distinct scenes.')
    uploaded = files.upload()
    image_items = list(uploaded.items())
    if not image_items:
        raise ValueError('Upload at least one image.')
else:
    raise ValueError("DATA_SOURCE must be 'coco' or 'upload'")

def decode_and_resize(raw_bytes, maximum_side=640):
    image = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError('An uploaded file is not a readable JPG/PNG image.')
    height, width = image.shape[:2]
    scale = min(1.0, maximum_side / max(height, width))
    new_width = max(64, (int(width * scale) // 32) * 32)
    new_height = max(64, (int(height * scale) // 32) * 32)
    return cv2.resize(image, (new_width, new_height))

# Split by source image, before any synthetic warps or descriptor calibration.
# Upload images of distinct scenes; near-duplicate photos belong to the same split.
image_items = sorted(image_items)
images = [decode_and_resize(raw) for _, raw in image_items]
if len(images) >= 2:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(images))
    validation_count = max(1, int(np.ceil(0.25 * len(images))))
    validation_ids = set(order[:validation_count].tolist())
    calibration_images = [im for i, im in enumerate(images) if i not in validation_ids]
    validation_images = [im for i, im in enumerate(images) if i in validation_ids]
    print('Calibration files:', [name for i, (name, _) in enumerate(image_items) if i not in validation_ids])
    print('Held-out files:', [name for i, (name, _) in enumerate(image_items) if i in validation_ids])
else:
    calibration_images = validation_images = images
    print('WARNING: One source image only; validation shares scene content with training.')
print('Calibration images:', len(calibration_images), 'Validation images:', len(validation_images))

The repository verlab_accelerated_features does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/verlab/accelerated_features/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/verlab/accelerated_features/raw/main/weights/xfeat.pt" to /root/.cache/torch/hub/checkpoints/xfeat.pt


100%|██████████| 5.96M/5.96M [00:00<00:00, 129MB/s]


Device: cpu
Mounted at /content/drive
Persistent dataset cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets
Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/val2017.zip
Selected 24 COCO images using seed 7; no COCO annotations needed.
Calibration files: ['000000002261.jpg', '000000029675.jpg', '000000067534.jpg', '000000162858.jpg', '000000165713.jpg', '000000173008.jpg', '000000175364.jpg', '000000194506.jpg', '000000270402.jpg', '000000289343.jpg', '000000360097.jpg', '000000460841.jpg', '000000471869.jpg', '000000479596.jpg', '000000504711.jpg', '000000517832.jpg', '000000528524.jpg', '000000547383.jpg']
Held-out files: ['000000077595.jpg', '000000132544.jpg', '000000334006.jpg', '000000394559.jpg', '000000449432.jpg', '000000474039.jpg']
Calibration images: 18 Validation images: 6


## Shared features and calibration split

In [3]:
def image_to_tensor(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()[None].to(device) / 255.0

@torch.inference_mode()
def extract_features(image):
    output = xfeat.detectAndCompute(image_to_tensor(image), top_k=TOP_K)[0]
    return {
        'keypoints': output['keypoints'].detach().cpu().float(),
        'descriptors': output['descriptors'].detach().cpu().float(),
    }

def make_random_homography(height, width, generator):
    center = (width / 2.0, height / 2.0)
    angle = generator.uniform(-20.0, 20.0)
    scale = generator.uniform(0.88, 1.08)
    affine = cv2.getRotationMatrix2D(center, angle, scale)
    affine[0, 2] += generator.uniform(-0.06, 0.06) * width
    affine[1, 2] += generator.uniform(-0.06, 0.06) * height
    homography = np.vstack([affine, [0., 0., 1.]]).astype(np.float32)
    homography[2, 0] = generator.uniform(-7e-5, 7e-5)
    homography[2, 1] = generator.uniform(-7e-5, 7e-5)
    return homography

def warp_image(image, homography, generator):
    height, width = image.shape[:2]
    warped = cv2.warpPerspective(image, homography, (width, height),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    alpha = float(generator.uniform(0.82, 1.12))
    beta = float(generator.uniform(-12., 18.))
    return np.clip(warped.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

def project_points(points, homography):
    p = points.detach().cpu().numpy()
    p_h = np.concatenate([p, np.ones((len(p), 1), np.float32)], axis=1)
    projected = (homography @ p_h.T).T
    return torch.from_numpy((projected[:, :2] / projected[:, 2:3]).astype(np.float32))

def ground_truth_pairs(keypoints_1, keypoints_2, homography):
    if not len(keypoints_1) or not len(keypoints_2):
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    distances_12 = torch.cdist(project_points(keypoints_1, homography), keypoints_2)
    nearest_12 = distances_12.argmin(dim=1)
    errors = distances_12[torch.arange(len(keypoints_1)), nearest_12]
    distances_21 = torch.cdist(project_points(keypoints_2, np.linalg.inv(homography)), keypoints_1)
    nearest_21 = distances_21.argmin(dim=1)
    indices = torch.arange(len(keypoints_1))
    good = (nearest_21[nearest_12] == indices) & (errors < GEOMETRIC_THRESHOLD)
    return indices[good], nearest_12[good]

def build_pairs(source_images, warps_per_image, generator):
    pairs = []
    for image_index, image in enumerate(source_images):
        base = extract_features(image)
        for warp_index in range(warps_per_image):
            h = make_random_homography(*image.shape[:2], generator)
            warped_image = warp_image(image, h, generator)
            warped = extract_features(warped_image)
            i, j = ground_truth_pairs(base['keypoints'], warped['keypoints'], h)
            if len(i) > MAX_POSITIVES_PER_PAIR:
                keep = torch.randperm(len(i))[:MAX_POSITIVES_PER_PAIR]
                i, j = i[keep], j[keep]
            if len(i) < 32:
                print(f'Warning: image {image_index}, warp {warp_index}: {len(i)} positive pairs')
            pairs.append({'d1': base['descriptors'], 'd2': warped['descriptors'],
                          'k1': base['keypoints'], 'k2': warped['keypoints'],
                          'pos1': i, 'pos2': j, 'H': h, 'image1': image, 'image2': warped_image})
    return pairs

start = time.perf_counter()
calibration_pairs = build_pairs(calibration_images, CALIBRATION_WARPS_PER_IMAGE, rng)
validation_pairs = build_pairs(validation_images, VALIDATION_WARPS_PER_IMAGE, rng)
calibration = torch.cat([torch.cat([p['d1'], p['d2']], dim=0) for p in calibration_pairs], dim=0)
if len(calibration) > 50000:
    calibration = calibration[torch.randperm(len(calibration))[:50000]]
print('Feature extraction time (calibration and validation):', round(time.perf_counter() - start, 2), 's')
print('Calibration pairs:', len(calibration_pairs), 'Validation pairs:', len(validation_pairs),
      'Calibration descriptors:', len(calibration))

Feature extraction time (calibration and validation): 39.32 s
Calibration pairs: 108 Validation pairs: 18 Calibration descriptors: 50000


In [4]:
real_pairs = []
if RUN_HPatches:
    HPATCHES_URL = 'https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'
    HPATCHES_SEQUENCES_PER_TYPE = 6  # 6 viewpoint + 6 illumination scenes
    HPATCHES_TARGETS = (2, 3)       # two real photos per sequence
    hpatches_zip = data_dir / 'hpatches-sequences-release.zip'
    download_archive(HPATCHES_URL, hpatches_zip)

    def decode_hpatches_with_scale(raw_bytes):
        original = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
        if original is None:
            raise RuntimeError('Failed to decode HPatches PPM image')
        old_h, old_w = original.shape[:2]
        resized = decode_and_resize(raw_bytes)
        new_h, new_w = resized.shape[:2]
        scale = np.diag([new_w / old_w, new_h / old_h, 1.0]).astype(np.float64)
        return resized, scale

    real_pairs = []
    with zipfile.ZipFile(hpatches_zip) as archive:
        entries = set(archive.namelist())
        folders = sorted({name.rsplit('/', 1)[0] for name in entries if name.endswith('/1.ppm')})
        for prefix in ('v_', 'i_'):
            selected_folders = [folder for folder in folders if folder.split('/')[-1].startswith(prefix)]
            if len(selected_folders) < HPATCHES_SEQUENCES_PER_TYPE:
                raise RuntimeError(f'Expected more HPatches {prefix} folders; found {len(selected_folders)}')
            for folder in selected_folders[:HPATCHES_SEQUENCES_PER_TYPE]:
                image_1, scale_1 = decode_hpatches_with_scale(archive.read(folder + '/1.ppm'))
                features_1 = extract_features(image_1)
                for target_id in HPATCHES_TARGETS:
                    image_name = f'{folder}/{target_id}.ppm'
                    h_name = f'{folder}/H_1_{target_id}'
                    if image_name not in entries or h_name not in entries:
                        raise RuntimeError(f'Missing HPatches image or homography: {folder}, {target_id}')
                    image_2, scale_2 = decode_hpatches_with_scale(archive.read(image_name))
                    features_2 = extract_features(image_2)
                    original_h = np.loadtxt(io.BytesIO(archive.read(h_name))).reshape(3, 3)
                    resized_h = scale_2 @ original_h @ np.linalg.inv(scale_1)
                    real_pairs.append({
                        'name': f'{folder.split("/")[-1]}:1->{target_id}',
                        'group': prefix, 'd1': features_1['descriptors'],
                        'd2': features_2['descriptors'],
                        'k1': features_1['keypoints'], 'k2': features_2['keypoints'],
                        'H': resized_h, 'image1': image_1, 'image2': image_2,
                    })
    print('HPatches real-photo pairs:', len(real_pairs), '(viewpoint and illumination)')



Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/hpatches-sequences-release.zip
HPatches real-photo pairs: 24 (viewpoint and illumination)


In [5]:
import inspect, hashlib
OFFICIAL_MIN_COSSIM = -1  # match_xfeat default, NOT standalone match default 0.82
signature=inspect.signature(xfeat.match_xfeat)
if signature.parameters['min_cossim'].default != OFFICIAL_MIN_COSSIM:
    raise RuntimeError('Official match_xfeat default changed; review comparison settings.')
official_source=inspect.getsource(type(xfeat).match)
extract_source=inspect.getsource(type(xfeat).detectAndCompute)
hub_path=Path(inspect.getfile(type(xfeat))).resolve().parent.parent
sha_run=subprocess.run(['git','-C',str(hub_path),'rev-parse','HEAD'],capture_output=True,text=True)
xfeat_revision=sha_run.stdout.strip() if sha_run.returncode==0 else None
xfeat_provenance=dict(hub_path=str(hub_path),git_revision=xfeat_revision,
    match_sha256=hashlib.sha256(official_source.encode()).hexdigest(),
    extract_sha256=hashlib.sha256(extract_source.encode()).hexdigest(),
    match_signature=str(inspect.signature(xfeat.match)),match_xfeat_signature=str(signature))
print('Official XFeat provenance:',xfeat_provenance)
print('Official matcher source SHA256:',xfeat_provenance['match_sha256'])

@torch.inference_mode()
def official_match_cpu(p):
    # Direct call, no replacement of matmul, max, MNN or threshold logic.
    i,j=xfeat.match(p['d1'],p['d2'],min_cossim=OFFICIAL_MIN_COSSIM)
    return i.cpu().numpy(),j.cpu().numpy()



Official XFeat provenance: {'hub_path': '/root/.cache/torch/hub/verlab_accelerated_features_main', 'git_revision': None, 'match_sha256': 'e987220ff12fe3352c01ce2d411cdf70f2756f4039398eb2e98f62dfdcd0ad14', 'extract_sha256': '810bafd27792ba820c54ca66a11cd6eb243384641419436b39aeff5835ae9308', 'match_signature': '(feats1, feats2, min_cossim=0.82)', 'match_xfeat_signature': '(img1, img2, top_k=None, min_cossim=-1)'}
Official matcher source SHA256: e987220ff12fe3352c01ce2d411cdf70f2756f4039398eb2e98f62dfdcd0ad14


In [6]:
import platform, json
CPU_THREADS=NUMBA_THREADS
AVX2=None
METHOD='official XFeat cosine MNN (-1)'
DESCRIPTOR_BYTES=256
def prepare(p):return p
def cached_match(p):return official_match_cpu(p)
def total_match(p):return official_match_cpu(p)
@torch.inference_mode()
def pipeline(t1,t2):return xfeat.match_xfeat(t1,t2,top_k=TOP_K,min_cossim=OFFICIAL_MIN_COSSIM)


## Accuracy and speed for this method only

In [7]:
import pandas as pd
datasets={'synthetic':[prepare(p) for p in validation_pairs]}
if real_pairs:datasets['HPatches']=[prepare(p) for p in real_pairs]
accuracy_results=[];pair_accuracy=[];pair_manifest=[]
for label,pairs in datasets.items():
 errors=[];mutual=correct=0
 for idx,p in enumerate(pairs):
  i,j=cached_match(p)
  if BINS is not None:
   rd=pairwise_uint8_abs(variant.codes(p['x']),variant.codes(p['y']))
   np.testing.assert_array_equal(rd,variant.matrix(p['pa'],variant.encode(p['y'])))
   for u,v in zip(mutual_from_scores(rd,False),(i,j)):np.testing.assert_array_equal(u,v)
  err=np.linalg.norm(project_points(p['k1'][i],p['H']).numpy()-p['k2'][j].numpy(),axis=1)
  good=int((err<GEOMETRIC_THRESHOLD).sum());mutual+=len(i);correct+=good;errors.extend(err.tolist())
  pair_accuracy.append(dict(dataset=label,pair=idx,method=METHOD,mutual=len(i),correct=good,
   precision_pct=100*good/len(i) if len(i) else None,median_px=float(np.median(err)) if len(err) else None))
  pair_manifest.append(dict(dataset=label,pair=idx,name=p.get('name',f'synthetic_{idx}'),
   n=len(p['d1']),m=len(p['d2']),homography=np.asarray(p['H']).tolist(),
   descriptor_sha256=hashlib.sha256(p['d1'].numpy().tobytes()+p['d2'].numpy().tobytes()).hexdigest()))
 accuracy_results.append(dict(dataset=label,method=METHOD,pairs=len(pairs),mutual=mutual,correct=correct,
  precision_pct=100*correct/mutual if mutual else None,median_px=float(np.median(errors)) if errors else None,
  descriptor_bytes=DESCRIPTOR_BYTES,bins=BINS,thermometer_bits=BINS-1 if BINS else None,
  exact_abs_verified=True if BINS else None))
display(pd.DataFrame(accuracy_results))

# Each scope is measured directly. Keep all raw samples to inspect runtime noise.
def bench(fn):
 for _ in range(WARMUP):fn()
 samples=[]
 for _ in range(REPEATS):
  start=time.perf_counter_ns();fn();samples.append((time.perf_counter_ns()-start)/1e6)
 return dict(median_ms=float(np.median(samples)),p10_ms=float(np.percentile(samples,10)),
  p90_ms=float(np.percentile(samples,90)),samples_ms=samples)
@torch.inference_mode()
def extraction_only(t1,t2):return xfeat.detectAndCompute(t1,top_k=TOP_K)[0],xfeat.detectAndCompute(t2,top_k=TOP_K)[0]
def coordinate_error(xy1,xy2,H):
 if not len(xy1):return np.empty(0)
 xyh=np.c_[xy1,np.ones(len(xy1))];proj=(H@xyh.T).T;proj=proj[:,:2]/proj[:,2:]
 return np.linalg.norm(proj-xy2,axis=1)
timing_results=[];pipeline_results=[];stage_results=[]
for label,pairs in datasets.items():
 indices=np.unique(np.linspace(0,len(pairs)-1,min(TIMING_PAIRS_PER_DATASET,len(pairs)),dtype=int))
 for idx in indices:
  p=pairs[idx];t1=image_to_tensor(p['image1']);t2=image_to_tensor(p['image2'])
  d1,d2=extraction_only(t1,t2)
  if BINS:
   i,j=variant.full(dict(x=d1['descriptors'].numpy(),y=d2['descriptors'].numpy()))
  else:i,j=xfeat.match(d1['descriptors'],d2['descriptors'],min_cossim=OFFICIAL_MIN_COSSIM)
  xy=pipeline(t1,t2)
  np.testing.assert_array_equal(xy[0],d1['keypoints'][i].numpy());np.testing.assert_array_equal(xy[1],d2['keypoints'][j].numpy())
  # Randomize scope order without touching the dataset RNG.
  funcs={'cached':lambda:cached_match(p),'total':lambda:total_match(p),
   'extraction':lambda:extraction_only(t1,t2),'pipeline':lambda:pipeline(t1,t2)}
  order=list(funcs);np.random.default_rng(SEED+int(idx)).shuffle(order)
  stats={scope:bench(funcs[scope]) for scope in order}
  timing_results.append(dict(dataset=label,pair=int(idx),method=METHOD,n=len(p['d1']),m=len(p['d2']),
   cached_ms=stats['cached']['median_ms'],total_ms=stats['total']['median_ms'],
   p10_ms=stats['total']['p10_ms'],p90_ms=stats['total']['p90_ms'],
   cached_samples_ms=stats['cached']['samples_ms'],total_samples_ms=stats['total']['samples_ms']))
  err=coordinate_error(xy[0],xy[1],p['H'])
  pipeline_results.append(dict(dataset=label,pair=int(idx),method=METHOD,
   extraction_only_ms=stats['extraction']['median_ms'],pipeline_ms=stats['pipeline']['median_ms'],
   p10_ms=stats['pipeline']['p10_ms'],p90_ms=stats['pipeline']['p90_ms'],
   mutual=len(xy[0]),correct=int((err<GEOMETRIC_THRESHOLD).sum()),
   precision_pct=100*int((err<GEOMETRIC_THRESHOLD).sum())/len(err) if len(err) else None,
   samples_ms=stats['pipeline']['samples_ms'],extraction_samples_ms=stats['extraction']['samples_ms']))
  if BINS:
   encoding=bench(lambda:(variant.encode(p['x']),variant.encode(p['y'])))
   layout=bench(lambda:variant.soa(variant.encode(p['y'])))  # encoding + layout, explicitly labeled
   stage_results.append(dict(dataset=label,pair=int(idx),both_sets_encoding_ms=encoding['median_ms'],
    B_encoding_and_layout_ms=layout['median_ms']))
  print('Timed:',label,int(idx), '| total:',round(stats['total']['median_ms'],3),
   'ms | pipeline:',round(stats['pipeline']['median_ms'],3),'ms')
speed_summary=pd.DataFrame(timing_results).groupby(['dataset','method'],sort=False)[['cached_ms','total_ms','p10_ms','p90_ms']].median().reset_index()
pipeline_summary=pd.DataFrame(pipeline_results).groupby(['dataset','method'],sort=False)[['extraction_only_ms','pipeline_ms','p10_ms','p90_ms']].median().reset_index()
print('Matching: extraction excluded; thermometer total includes encoding and layout.')
display(speed_summary.round(4))
print('Pipeline: official extraction twice + selected matcher + coordinate output. Image I/O and tensor preparation excluded.')
display(pipeline_summary.round(4))


,dataset,method,pairs,mutual,correct,precision_pct,median_px,descriptor_bytes,bins,thermometer_bits,exact_abs_verified
0,synthetic,official XFeat cosine MNN (-1),18,8200,5611,68.426829,1.366410,256,None,None,None
1,HPatches,official XFeat cosine MNN (-1),24,10607,7138,67.295182,1.176445,256,None,None,None


Timed: synthetic 0 | total: 3.757 ms | pipeline: 194.011 ms
Timed: synthetic 4 | total: 3.731 ms | pipeline: 195.424 ms
Timed: synthetic 8 | total: 3.777 ms | pipeline: 195.903 ms
Timed: synthetic 12 | total: 3.8 ms | pipeline: 191.319 ms
Timed: synthetic 17 | total: 3.773 ms | pipeline: 180.985 ms
Timed: HPatches 0 | total: 15.909 ms | pipeline: 225.581 ms
Timed: HPatches 5 | total: 3.764 ms | pipeline: 210.26 ms
Timed: HPatches 11 | total: 3.896 ms | pipeline: 227.036 ms
Timed: HPatches 17 | total: 8.383 ms | pipeline: 182.394 ms
Timed: HPatches 23 | total: 4.2 ms | pipeline: 156.161 ms
Matching: extraction excluded; thermometer total includes encoding and layout.


,dataset,method,cached_ms,total_ms,p10_ms,p90_ms
0,synthetic,official XFeat cosine MNN (-1),3.8603,3.7726,3.7297,3.8412
1,HPatches,official XFeat cosine MNN (-1),3.8367,4.1995,3.8871,4.6540


Pipeline: official extraction twice + selected matcher + coordinate output. Image I/O and tensor preparation excluded.


,dataset,method,extraction_only_ms,pipeline_ms,p10_ms,p90_ms
0,synthetic,official XFeat cosine MNN (-1),196.1522,194.0109,188.5322,205.6533
1,HPatches,official XFeat cosine MNN (-1),206.9755,210.2601,206.1352,243.4036


## Save speed, accuracy and reproducibility records

In [8]:
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
run_id=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
result_dir=RESULT_ROOT/EXPERIMENT/run_id
result_dir.mkdir(parents=True,exist_ok=True)
pd.DataFrame(accuracy_results).to_csv(result_dir/'accuracy.csv',index=False)
pd.DataFrame(pair_accuracy).to_csv(result_dir/'accuracy_per_pair.csv',index=False)
speed_summary.to_csv(result_dir/'speed_summary.csv',index=False)
pipeline_summary.to_csv(result_dir/'pipeline_summary.csv',index=False)
pd.DataFrame(stage_results).to_csv(result_dir/'encoding_stages.csv',index=False)
report=dict(experiment=EXPERIMENT,method=METHOD,run_id=run_id,seed=SEED,top_k=TOP_K,
 bins=BINS,thermometer_bits=BINS-1 if BINS else None,descriptor_bytes=DESCRIPTOR_BYTES,
 threads=CPU_THREADS,warmup=WARMUP,repeats=REPEATS,avx2_required=bool(BINS),
 platform=platform.platform(),cpu_info=subprocess.run(['lscpu'],capture_output=True,text=True).stdout,
 versions=dict(numpy=np.__version__,torch=torch.__version__,opencv=cv2.__version__),
 accuracy=accuracy_results,pair_accuracy=pair_accuracy,timing=timing_results,pipeline=pipeline_results,
 encoding_stages=stage_results,pair_manifest=pair_manifest,xfeat_provenance=xfeat_provenance,
 official_min_cossim=OFFICIAL_MIN_COSSIM,
 calibration_files=[name for k,(name,_) in enumerate(image_items) if k not in validation_ids] if len(images)>=2 else [name for name,_ in image_items],
 validation_files=[name for k,(name,_) in enumerate(image_items) if k in validation_ids] if len(images)>=2 else [name for name,_ in image_items],
 note='Precision: pooled matches with forward homography error <3px in resized coordinates. Match total includes thermometer encoding/layout; extraction excluded. Full pipeline measured directly. Calibration and compilation excluded. Compare separate runs only after checking CPU, versions, fingerprints and timing spread.')
(result_dir/'results.json').write_text(json.dumps(report,indent=2,allow_nan=False))
(result_dir/'official_match_source.py').write_text(official_source)
if BINS:np.savez(result_dir/'fixed_edges.npz',edges=variant.edges)
print('Saved speed and accuracy:',result_dir)


Saved speed and accuracy: /content/drive/MyDrive/Colab Notebooks/success-direct-thermometer-AVX2-SIMD4/results/official_XFeat/20261008_143431_468387
